# 02 · Agent From Scratch (the control plane, hand-rolled)

**Where we are in the stack:** the **Agent = control plane** (Layer 4). We build the loop *by hand* so you can see the state machine before a framework hides it.

The loop (a protocol FSM for tool use):

```
prompt the model  ->  did it ask for a tool?
      ^                       |  yes                 |  no
      |                  call the tool          FINAL ANSWER -> stop
      |                       |
      +----- feed result back +
```

`max_iterations` is the **TTL / hop count** - it stops a runaway loop.

> Note: tool-calling needs a tool-capable model (e.g. `gpt-4o-mini`, or local `qwen2.5` / `llama3.1`). A 1B model usually cannot do this reliably.

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. Define the tools (plain Python functions)

Two flavours, mirroring real ops work:
- `calculate_subnet` - pure compute, deterministic.
- `get_interface_status` - I/O to the "world" (mocked here; wire it to netmiko / gNMI / your MCP server in real life).

In [ ]:
import ipaddress, hashlib, json

def calculate_subnet(cidr):
    '''Compute network, broadcast, netmask and usable host count for a CIDR.'''
    net = ipaddress.ip_network(cidr, strict=False)
    if net.version == 4:
        usable = net.num_addresses - 2 if net.prefixlen <= 30 else net.num_addresses
    else:
        usable = net.num_addresses
    return {
        "network": str(net.network_address),
        "broadcast": str(net.broadcast_address) if net.version == 4 else "n/a",
        "netmask": str(net.netmask),
        "prefix_length": net.prefixlen,
        "total_addresses": net.num_addresses,
        "usable_hosts": usable,
    }

def get_interface_status(device, interface):
    '''MOCK telemetry. In production wire this to netmiko / SNMP / gNMI / your MCP server.'''
    h = int(hashlib.md5(f"{device}{interface}".encode()).hexdigest(), 16)
    up = (h % 5 != 0)  # ~80 percent up, deterministic so demos are repeatable
    return {
        "device": device, "interface": interface,
        "admin_status": "up",
        "oper_status": "up" if up else "down",
        "speed": "10Gbps", "mtu": 1500,
        "input_errors": h % 7, "output_errors": h % 3, "crc_errors": h % 4,
    }

In [ ]:
# Sanity-check the tools WITHOUT the model (prove they work before the LLM touches them)
print(calculate_subnet("10.20.0.0/22"))
print(get_interface_status("leaf-01", "ethernet1/0/1"))

## 2. Tell the model what tools exist (JSON schema)

The model never executes anything. It only *requests* a call by name + JSON args. **You** execute and feed the result back.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "calculate_subnet",
        "description": "Compute network, broadcast, netmask and usable host count for an IPv4/IPv6 CIDR.",
        "parameters": {"type": "object",
            "properties": {"cidr": {"type": "string", "description": "CIDR, e.g. 10.20.0.0/22"}},
            "required": ["cidr"]}}},
    {"type": "function", "function": {
        "name": "get_interface_status",
        "description": "Operational status and error counters for an interface on a device.",
        "parameters": {"type": "object",
            "properties": {
                "device":    {"type": "string", "description": "hostname, e.g. leaf-01"},
                "interface": {"type": "string", "description": "interface, e.g. ethernet1/0/1"}},
            "required": ["device", "interface"]}}},
]

# name -> callable. This is your capability table.
TOOL_REGISTRY = {
    "calculate_subnet": calculate_subnet,
    "get_interface_status": get_interface_status,
}

## 3. The agent loop (this ~35 lines IS the agent)

In [ ]:
SYSTEM = "You are a network operations assistant. Use tools when they help. Be concise."

def run_agent(user_query, max_iterations=5, temperature=0):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_query},
    ]
    print("USER:", user_query); print("=" * 64)

    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=temperature,
        )
        msg = resp.choices[0].message

        # TERMINATION: model asked for no tool -> it has a final answer.
        if not msg.tool_calls:
            print(f"[step {step}] FINAL ANSWER\n{msg.content}")
            return msg.content

        # Record the model's turn (its tool request), then execute each call.
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"[step {step}] TOOL CALL  -> {name}({args})")
            result = TOOL_REGISTRY[name](**args)            # <-- WE run it
            print(f"[step {step}] TOOL RESULT <- {result}")
            messages.append({                               # feed the result back
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return "Stopped: hit max_iterations (TTL expired)."  # loop guard

## 4. Run it - one tool

In [ ]:
_ = run_agent("How many usable hosts are in 172.16.4.0/26?")

## 5. Run it - multiple tools in one task (watch the FSM iterate)

In [ ]:
_ = run_agent(
    "Is interface ethernet1/0/1 on leaf-01 operational, "
    "and how many usable hosts are in 10.20.0.0/22?"
)

## 6. Run it - a reasoning chain across steps

In [ ]:
_ = run_agent(
    "Check whether et-0/0/1 on spine-02 is up. If it is up, also give me the "
    "broadcast address of 192.168.10.0/24. If it is down, just say it's down."
)

## Recap - and why you'd want a framework

You just built a real agent: **prompt -> request tool -> execute -> feed back -> repeat**, holding state. That is a control-plane state machine.

But notice what's missing the moment this goes to production:
- retries / timeouts / error recovery on tool failures
- parallel tool execution
- streaming partial output
- persistent memory + **checkpointing** (resume a run)
- observability (trace every transition)
- confirmation gates before *config-mutating* tools run (default read-only)

You don't hand-roll BGP - you run FRR. Same here.

**Next:** the identical agent via LangChain/LangGraph, where the loop is the framework's job. -> `03_agent_with_langchain.ipynb`